# 02 — google.colab.ai Wrapper (0 настройки)

Использует встроенный AI в Google Colab — без API ключа, без GPU.

**Требования:** Только Google Colab  
**GPU:** Не нужен  
**API ключ:** Не нужен  
**Ограничения:** Нет temperature/system_instruction (эмулируется через prompt)

In [ ]:
!pip install -q fastapi uvicorn openai

In [ ]:
import google.colab.ai as ai

models = ai.list_models()
print("Доступные модели:")
for m in models:
    print(f"  - {m}")

DEFAULT_MODEL = models[0] if models else "gemini"
print(f"\nИспользуем: {DEFAULT_MODEL}")

PORT = 8080

In [ ]:
import google.colab.ai as ai
from fastapi import FastAPI, Request
from fastapi.responses import JSONResponse
import uvicorn
import threading
import time
import uuid

app = FastAPI(title="Colab AI → OpenAI Proxy")

@app.get("/health")
async def health():
    return {"status": "ok", "model": DEFAULT_MODEL, "backend": "colab-ai"}

@app.post("/v1/chat/completions")
async def chat_completions(request: Request):
    body = await request.json()
    messages = body.get("messages", [])
    model_name = body.get("model", DEFAULT_MODEL)
    max_tokens = body.get("max_tokens", 200)

    # Build prompt from messages (colab.ai doesn't support chat format)
    prompt_parts = []
    for msg in messages:
        role = msg.get("role", "user")
        content = msg.get("content", "")
        if role == "system":
            prompt_parts.insert(0, f"Instructions: {content}")
        elif role == "assistant":
            prompt_parts.append(f"Assistant: {content}")
        else:
            prompt_parts.append(f"User: {content}")
    
    prompt_parts.append("Assistant:")
    full_prompt = "\n\n".join(prompt_parts)

    try:
        response_text = ai.generate_text(
            prompt=full_prompt,
            model=model_name if model_name not in ("local", "default", "gpt-3.5-turbo", "gpt-4") else DEFAULT_MODEL,
        )

        return JSONResponse({
            "id": f"chatcmpl-{uuid.uuid4().hex[:8]}",
            "object": "chat.completion",
            "model": model_name,
            "choices": [{
                "index": 0,
                "message": {"role": "assistant", "content": response_text},
                "finish_reason": "stop"
            }],
            "usage": {
                "prompt_tokens": 0,
                "completion_tokens": 0,
                "total_tokens": 0
            }
        })
    except Exception as e:
        return JSONResponse(
            status_code=500,
            content={"error": {"message": str(e), "type": "server_error"}}
        )

def run_server():
    uvicorn.run(app, host="0.0.0.0", port=PORT, log_level="warning")

thread = threading.Thread(target=run_server, daemon=True)
thread.start()
time.sleep(2)
print(f"✓ Сервер запущен на порту {PORT}")

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

import subprocess, re, time

process = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", f"http://localhost:{PORT}"],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True
)

tunnel_url = None
for _ in range(30):
    line = process.stderr.readline()
    match = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if match:
        tunnel_url = match.group(0)
        break
    time.sleep(1)

if tunnel_url:
    print(f"✓ Туннель активен!")
    print(f"\n{'='*60}")
    print(f"  URL: {tunnel_url}")
    print(f"  Health: {tunnel_url}/health")
    print(f"  API: {tunnel_url}/v1/chat/completions")
    print(f"{'='*60}")
else:
    print("✗ Не удалось получить URL туннеля")

In [ ]:
from openai import OpenAI

client = OpenAI(base_url=f"http://localhost:{PORT}/v1", api_key="not-needed")

response = client.chat.completions.create(
    model="default",
    messages=[
        {"role": "system", "content": """Extract from the invoice text: country, doc_type (electricity/telecom/bank/water/gas/tax/other), company, year.
Respond ONLY with JSON: {"country": "...", "doc_type": "...", "company": "...", "year": ...}"""},
        {"role": "user", "content": """RECHNUNG
Wiener Netze GmbH
Erdbergstraße 236, 1110 Wien
Rechnungsdatum: 15.03.2024
Stromrechnung für den Zeitraum 01.01.2024 - 31.03.2024
Gesamtbetrag: EUR 245,67 inkl. MwSt
UID: ATU12345678"""}
    ],
    max_tokens=200
)

print("Ответ модели:")
print(response.choices[0].message.content)

## Подключение к InvoiceLLM

Добавьте в `config.yaml`:

```yaml
llm:
  servers:
    - name: "Colab AI Wrapper"
      host: "YOUR-URL.trycloudflare.com"
      port: 443
      ssl: true
      priority: 1
```

**Ограничения:**
- Нет контроля temperature (всегда default)
- Нет system_instruction (эмулируется через prompt prefix)
- Набор моделей зависит от Colab